# Evaluating Coding Models on a Generated Benchmark

This notebook evaluates multiple coding models against the **Domain Code Evaluation
Benchmark** generated by the `domain-code-eval` flow (see `domain_code_eval.ipynb`).

The benchmark contains 42 execution-verified problems across 6 domains. Each problem
has a **problem description**, a **reference solution**, and a **test suite** — all
verified to execute correctly via the Monty sandboxed interpreter.

**Evaluation protocol:**
1. Give the model only the `problem_description`
2. Model generates a function implementation
3. Run the verified `test_code` against the model's code
4. pass@1 = fraction of problems where all tests pass

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

# pip install sdg_hub[code]        # pydantic-monty for sandboxed execution
# pip install sdg_hub[examples]    # notebook dependencies

In [ ]:
import warnings

import nest_asyncio
import pandas as pd

from sdg_hub.core.blocks.code import PythonInterpreterBlock
from sdg_hub.core.blocks.llm import LLMChatBlock, LLMResponseExtractorBlock

warnings.filterwarnings("ignore")
nest_asyncio.apply()

## 2. Load the Benchmark

In [ ]:
benchmark = pd.read_json("domain_code_benchmark.jsonl", lines=True)

print(f"Benchmark: {len(benchmark)} verified problems")
print(f"Domains: {sorted(benchmark['domain'].unique())}")
print(f"Difficulties: {sorted(benchmark['difficulty'].unique())}")
print("\nDistribution:")
print(benchmark.groupby(["domain", "difficulty"]).size().unstack(fill_value=0))

## 3. Define the Evaluation Harness

A reusable function that takes a model name, generates solutions for all benchmark
problems, and grades them by executing the test suites.

In [ ]:
SYSTEM_PROMPT = (
    "You are a Python programmer. Write the requested function. "
    "Output ONLY the Python function code. No explanations, no markdown fences, "
    "no import statements. The function must use only built-in Python features."
)


def evaluate_model(model_name: str, benchmark_df: pd.DataFrame) -> pd.DataFrame:
    """Evaluate a coding model on the benchmark.

    Returns a DataFrame with the original benchmark columns plus:
    - model: the model name
    - model_code: the model's generated code
    - passed: whether the test suite passed
    - error: error message if failed
    """
    df = benchmark_df.copy()

    # Step 1: Build prompts
    df["solve_prompt"] = df["problem_description"].apply(
        lambda p: [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p},
        ]
    )

    # Step 2: Generate solutions
    solver = LLMChatBlock(
        block_name=f"solve_{model_name.replace('.', '_').replace('-', '_')}",
        input_cols="solve_prompt",
        output_cols="raw_solution",
        model=model_name,
        max_tokens=2048,
        temperature=0.0,
        async_mode=True,
    )
    df = solver(df)

    # Step 3: Extract code from response
    extractor = LLMResponseExtractorBlock(
        block_name="extract_code",
        input_cols="raw_solution",
        extract_content=True,
        expand_lists=True,
    )
    df = extractor(df)

    # Step 4: Combine model code + test suite and execute
    df["candidate_code"] = df.apply(
        lambda row: str(row.get("extract_code_content", "")) + "\n\n" + row["test_code"],
        axis=1,
    )

    grader = PythonInterpreterBlock(
        block_name="grade",
        input_cols=["candidate_code"],
        output_cols=["grade_result"],
        interpreter_framework="monty",
        timeout=10.0,
    )
    df = grader(df)

    # Step 5: Extract results
    df["model"] = model_name
    df["model_code"] = df["extract_code_content"]
    df["passed"] = df["grade_result"].apply(lambda r: r.get("success", False))
    df["error"] = df["grade_result"].apply(
        lambda r: r.get("error") if not r.get("success") else None
    )

    return df[
        ["model", "domain", "difficulty", "function_spec", "passed", "error",
         "model_code", "problem_description", "function_code", "test_code"]
    ]


def print_results(results_df: pd.DataFrame):
    """Print a summary of evaluation results."""
    model = results_df["model"].iloc[0]
    total = len(results_df)
    passed = results_df["passed"].sum()
    print(f"\n{'='*70}")
    print(f"  {model}:  pass@1 = {passed}/{total} ({passed/total*100:.1f}%)")
    print(f"{'='*70}")

    print("\n  By domain:")
    for domain, group in results_df.groupby("domain"):
        p = group["passed"].sum()
        t = len(group)
        bar = "#" * p + "." * (t - p)
        print(f"    {domain:<28s}  {p:2d}/{t:2d}  [{bar}]")

    print("\n  By difficulty:")
    for diff in ["beginner", "intermediate", "advanced"]:
        group = results_df[results_df["difficulty"] == diff]
        if len(group) > 0:
            p = group["passed"].sum()
            t = len(group)
            print(f"    {diff:<15s}  {p:2d}/{t:2d}  ({p/t*100:.0f}%)")

## 4. Evaluate Models

Run each model on the full benchmark. Each evaluation generates solutions for all
42 problems, then grades them by executing the test suites in Monty's sandbox.

In [ ]:
models_to_evaluate = [
    "gpt-4.1-nano",
    "gpt-4.1-mini",
    "gpt-4o-mini",
]

all_results = []
for model in models_to_evaluate:
    print(f"\nEvaluating {model}...")
    result = evaluate_model(model, benchmark)
    print_results(result)
    all_results.append(result)

results_df = pd.concat(all_results, ignore_index=True)
print(f"\nTotal evaluations: {len(results_df)}")

## 5. Compare Models

In [ ]:
# Overall pass@1 comparison
print("Overall pass@1")
print("-" * 50)
summary = results_df.groupby("model")["passed"].agg(["sum", "count", "mean"])
summary.columns = ["passed", "total", "pass@1"]
summary = summary.sort_values("pass@1", ascending=False)
for model, row in summary.iterrows():
    bar = "#" * int(row["pass@1"] * 30) + "." * (30 - int(row["pass@1"] * 30))
    print(f"  {model:<20s}  {int(row['passed']):2d}/{int(row['total']):2d}  ({row['pass@1']*100:5.1f}%)  [{bar}]")

In [ ]:
# pass@1 by domain for each model
pivot_domain = results_df.groupby(["domain", "model"])["passed"].mean().unstack(fill_value=0)
pivot_domain = pivot_domain.round(2)

print("\npass@1 by Domain")
print("-" * 70)
print(pivot_domain.to_string())

# pass@1 by difficulty for each model
pivot_diff = results_df.groupby(["difficulty", "model"])["passed"].mean().unstack(fill_value=0)
pivot_diff = pivot_diff.reindex(["beginner", "intermediate", "advanced"]).round(2)

print("\n\npass@1 by Difficulty")
print("-" * 70)
print(pivot_diff.to_string())

## 6. Inspect Failures

Look at problems that differentiate the models — where one model passes and another
fails.

In [ ]:
# Find problems where models disagree
problem_pass = results_df.pivot_table(
    index="function_spec", columns="model", values="passed", aggfunc="first"
)

# Problems solved by all models
all_pass = problem_pass[problem_pass.all(axis=1)]
print(f"Solved by all models: {len(all_pass)}")

# Problems failed by all models
all_fail = problem_pass[~problem_pass.any(axis=1)]
print(f"Failed by all models: {len(all_fail)}")

# Discriminating problems (at least one model differs)
discriminating = problem_pass[problem_pass.nunique(axis=1) > 1]
print(f"Discriminating problems: {len(discriminating)}\n")

if len(discriminating) > 0:
    print("Problems that differentiate models:")
    print("-" * 90)
    for spec, row in discriminating.iterrows():
        results_str = "  ".join(
            f"{m}: {'PASS' if v else 'FAIL'}" for m, v in row.items()
        )
        print(f"  {spec[:55]:<57s} {results_str}")

In [ ]:
# Inspect a specific failure: show the problem, the model's code, and the error
failed = results_df[~results_df["passed"]]
if len(failed) > 0:
    ex = failed.iloc[0]
    print(f"Model: {ex['model']}")
    print(f"Problem: {ex['function_spec']}")
    print(f"Domain: {ex['domain']} | Difficulty: {ex['difficulty']}")
    print(f"\nError:\n{ex['error'][:300]}")
    print(f"\nModel's code (first 400 chars):\n{str(ex['model_code'])[:400]}")
else:
    print("No failures to inspect!")

## 7. Export Results

In [ ]:
# Export detailed results
# results_df.to_json("evaluation_results.jsonl", orient="records", lines=True)

# Summary table
summary_table = results_df.groupby("model").agg(
    problems=("passed", "count"),
    passed=("passed", "sum"),
    pass_at_1=("passed", "mean"),
).sort_values("pass_at_1", ascending=False)

summary_table["pass_at_1"] = (summary_table["pass_at_1"] * 100).round(1)
summary_table = summary_table.rename(columns={"pass_at_1": "pass@1 (%)"})
print("Final Leaderboard:")
summary_table

## 8. Performance Verification (LeetCode-style TLE detection)

Beyond correctness, we can check if solutions meet the required **time complexity**.
The benchmark includes an `input_generator` function for each problem. We run the
model's solution at increasing input sizes and check if timing scales as expected.

- **PASS** = correct and meets complexity
- **TLE** = correct but too slow (wrong complexity class)
- **FAIL** = incorrect (tests don't pass)

In [ ]:
import time

import pydantic_monty


def time_solution(function_code: str, input_generator: str, sizes: list[int],
                  timeout: float = 10.0) -> dict:
    """Run a solution at multiple input sizes and measure timing.

    Returns dict with sizes as keys and elapsed_ms as values.
    """
    timings = {}
    for n in sizes:
        code = f"{function_code}\n\n{input_generator}\n\nargs = generate_input({n})\n"
        # Find the function name from the code
        for line in function_code.strip().split("\n"):
            if line.startswith("def ") and "(" in line:
                func_name = line.split("(")[0].replace("def ", "").strip()
                break
        else:
            func_name = "unknown"

        if isinstance(code, str) and "class " in function_code:
            # Skip class-based problems for timing (need different invocation)
            return {}

        code += f"result = {func_name}(*args) if isinstance(args, tuple) else {func_name}(args)\n"

        m = pydantic_monty.Monty(code)
        t0 = time.perf_counter()
        try:
            m.run(
                limits=pydantic_monty.ResourceLimits(max_duration_secs=timeout),
                print_callback=lambda _f, _c: None,
            )
            elapsed = (time.perf_counter() - t0) * 1000
            timings[n] = elapsed
        except pydantic_monty.MontyError:
            elapsed = (time.perf_counter() - t0) * 1000
            timings[n] = None  # TLE or error
    return timings


def classify_complexity(timings: dict) -> str:
    """Classify observed complexity from timing data.

    Uses ratio of timing at largest vs smallest size to estimate.
    """
    sizes = sorted(k for k, v in timings.items() if v is not None)
    if len(sizes) < 2:
        return "unknown"

    t_small = timings[sizes[0]]
    t_large = timings[sizes[-1]]
    n_small = sizes[0]
    n_large = sizes[-1]

    if t_small < 0.01:  # Too fast to measure
        return "O(1) or too fast"

    ratio = t_large / t_small
    size_ratio = n_large / n_small

    # Expected ratios for different complexities (size_ratio = 50x for 100 -> 5000)
    # O(1):       ratio ~ 1
    # O(n):       ratio ~ 50
    # O(n log n): ratio ~ 50 * log(5000)/log(100) ~ 92
    # O(n^2):     ratio ~ 2500

    if ratio < 5:
        return "~O(1)"
    elif ratio < size_ratio * 2.5:
        return "~O(n)"
    elif ratio < size_ratio * 5:
        return "~O(n log n)"
    elif ratio < size_ratio ** 1.7:
        return "~O(n^2)"
    else:
        return "~O(n^2+)"


print("Timing verification functions defined")

In [ ]:
# Run timing verification on problems that have input generators
# (requires the updated benchmark with input_generator column)
benchmark_with_gen = pd.read_json("domain_code_benchmark.jsonl", lines=True)

if "input_generator" not in benchmark_with_gen.columns:
    print("Benchmark does not have input_generator column.")
    print("Re-run domain_code_eval.ipynb with the updated flow to generate it.")
else:
    # Filter to problems with valid input generators
    has_gen = benchmark_with_gen[
        benchmark_with_gen["input_generator"].notna()
        & (benchmark_with_gen["input_generator"].str.strip() != "")
        & (~benchmark_with_gen["function_code"].str.contains("class "))
    ].copy()

    print(f"Problems with input generators: {len(has_gen)}/{len(benchmark_with_gen)}")

    sizes = [100, 500, 1000, 5000]
    timing_results = []

    for idx, row in has_gen.iterrows():
        timings = time_solution(row["function_code"], row["input_generator"], sizes)
        if timings and all(v is not None for v in timings.values()):
            observed = classify_complexity(timings)
            timing_results.append({
                "function_spec": row["function_spec"],
                "domain": row["domain"],
                "expected": row.get("time_complexity", "?"),
                "observed": observed,
                **{f"t_{s}": f"{timings[s]:.1f}ms" for s in sizes},
            })

    if timing_results:
        timing_df = pd.DataFrame(timing_results)
        print(f"\nTiming results for {len(timing_df)} problems:")
        print(f"(input sizes: {sizes})\n")
        for _, row in timing_df.iterrows():
            match = "  " if row["expected"] == "?" else "OK" if row["observed"].strip("~") in row["expected"] else "??"
            print(
                f"  [{match}] {row['function_spec'][:45]:<47s} "
                f"expected={row['expected']:<15s} observed={row['observed']:<15s} "
                f"({row['t_100']:>8s} -> {row['t_5000']:>8s})"
            )